In [16]:
!pip install gradio youtube-search-python pandas scikit-learn --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.1 MB/s eta 0:00:00


In [20]:
# ---------------------------
# Step 1: Install required packages
# ---------------------------
!pip install gradio pandas scikit-learn --quiet

# ---------------------------
# Step 2: Import libraries
# ---------------------------
import gradio as gr
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------
# Step 3: Sample Movie Dataset
# ---------------------------
data = pd.DataFrame({
    "Movie": [
        "Inception", "Interstellar", "The Dark Knight",
        "Avatar", "Titanic", "The Matrix",
        "John Wick", "Mad Max Fury Road", "Gladiator"
    ],
    "Genre": [
        "Sci-Fi Action", "Sci-Fi Drama", "Action Crime",
        "Sci-Fi Adventure", "Romance Drama", "Sci-Fi Action",
        "Action Thriller", "Action Adventure", "Historical Action"
    ],
    "Poster": [
        "https://m.media-amazon.com/images/I/51s+Fq3pT-L._AC_.jpg",
        "https://m.media-amazon.com/images/I/71nFZGb9qUL._AC_SY679_.jpg",
        "https://m.media-amazon.com/images/I/51K8ouYrHeL._AC_.jpg",
        "https://m.media-amazon.com/images/I/41kTVLeW1CL._AC_.jpg",
        "https://m.media-amazon.com/images/I/51J0e9sQG8L._AC_.jpg",
        "https://m.media-amazon.com/images/I/51EG732BV3L._AC_.jpg",
        "https://m.media-amazon.com/images/I/51pWj3g9PbL._AC_.jpg",
        "https://m.media-amazon.com/images/I/51G8l7q4+ML._AC_.jpg",
        "https://m.media-amazon.com/images/I/51A2dI+9JlL._AC_.jpg"
    ]
})

# ---------------------------
# Step 4: Vectorize Genres & Compute Similarity
# ---------------------------
vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(data["Genre"])
similarity = cosine_similarity(genre_matrix)

# ---------------------------
# Step 5: Generate YouTube search links
# ---------------------------
def get_youtube_link(movie_name):
    return f"https://www.youtube.com/results?search_query={movie_name.replace(' ', '+')}+trailer"

# ---------------------------
# Step 6: Recommendation Function
# ---------------------------
def recommend_movie(movie_name):
    if movie_name not in data["Movie"].values:
        return "❌ Movie not found in dataset!"

    idx = data[data.Movie == movie_name].index[0]
    scores = list(enumerate(similarity[idx]))
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]

    rec_movies = []
    for i in sorted_scores:
        m = data.iloc[i[0]]["Movie"]
        poster = data.iloc[i[0]]["Poster"]
        link = get_youtube_link(m)
        rec_movies.append({
            "title": f"🎬 {m}",
            "poster": poster,
            "trailer": link
        })
    return rec_movies

# ---------------------------
# Step 7: Gradio GUI Function
# ---------------------------
def display_recommendations(movie):
    recs = recommend_movie(movie)
    if isinstance(recs, str):
        return recs  # error message
    # Build list of Markdown with poster and trailer button
    output_md = ""
    for r in recs:
        output_md += f"### {r['title']}\n"
        output_md += f"![poster]({r['poster']})\n\n"
        output_md += f"[Watch Trailer]({r['trailer']})\n\n"
    return output_md

# ---------------------------
# Step 8: Launch Gradio GUI
# ---------------------------
movie_list = list(data["Movie"])

gui = gr.Interface(
    fn=display_recommendations,
    inputs=gr.Dropdown(choices=movie_list, label="Select a Movie"),
    outputs=gr.Markdown(label="Recommended Movies with Posters & Trailers"),
    title="🎥 AI Movie Recommendation System",
    description="Select a movie to get top 5 similar movies with posters and YouTube trailer links."
)

# share=True for Colab
gui.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1a6bda140f838b8895.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://1a6bda140f838b8895.gradio.live
